# 04 · Medir cuánto cambia la correlación al excluir imputados · Resolución

Solución comentada de esta práctica de livecoding.

Seleccioná el kernel de `.venv` y usá **Reiniciar kernel y ejecutar todo**. La sección 3 contiene la función resuelta y la sección 4 ejecuta las pruebas de la práctica junto con la comprobación sobre datos reales. El resultado completo queda en la variable `resultado`.

La carga reutiliza `practicas_livecoding/preparacion.py` y los datasets ya descargados en `practicas_livecoding/datos/`. Conservá ambas carpetas dentro del mismo proyecto.

## 1. Preparación lista

La base equivale a `df` del notebook original: dormitorios imputados, marca de imputación y tres ratios. No se aplicó el filtro de curado.

Los archivos de `practicas_livecoding/datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_viviendas()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 20,640 | Columnas: 14


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,bedrooms_imputados,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,False,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,False,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,False,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,False,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,False,6.281853,0.172096,2.181467


## 2. Consigna y resultado esperado

Calculá la correlación de Pearson de cada ratio con `median_house_value`:
primero con todas las filas y después con aquellas donde `bedrooms_imputados` sea `False`.

Devolvé un DataFrame con índice llamado `variable`, en este orden:
`rooms_per_household`, `bedrooms_per_room`, `population_per_household`.
Las columnas deben ser `pearson_base`, `pearson_sin_imputados` y `diferencia`,
donde **`diferencia = pearson_sin_imputados - pearson_base`**.

Conservá `NaN` cuando una correlación no esté definida, por ejemplo si una variable
es constante. No redondees, no reemplaces esos valores por cero y no modifiques la entrada.

## 3. Resolución comentada

1. Construimos dos bases: todas las filas y solo las que no tuvieron imputación.
2. Calculamos sus matrices de Pearson y seleccionamos la columna del precio para los tres ratios.
3. Restamos `pearson_sin_imputados - pearson_base`, sin redondear.

Una diferencia positiva indica que el coeficiente aumentó al excluir imputados;
no demuestra causalidad ni que una de las dos estimaciones sea la verdadera.
Una correlación con una variable constante queda como `NaN`.

In [3]:
def comparar_correlaciones(datos):
    """Compara Pearson con todas las filas y con las filas sin imputaciones."""
    variables = [
        "rooms_per_household",
        "bedrooms_per_room",
        "population_per_household",
    ]
    columnas = variables + ["median_house_value"]
    sin_imputados = datos.loc[~datos["bedrooms_imputados"]]

    # Elegimos de cada matriz las correlaciones de los ratios con el precio.
    pearson_base = datos[columnas].corr()["median_house_value"].loc[variables]
    pearson_sin = sin_imputados[columnas].corr()["median_house_value"].loc[variables]

    resultado = pd.DataFrame({
        "pearson_base": pearson_base,
        "pearson_sin_imputados": pearson_sin,
    })
    resultado["diferencia"] = (
        resultado["pearson_sin_imputados"] - resultado["pearson_base"]
    )
    resultado.index.name = "variable"

    # Las correlaciones indefinidas permanecen como NaN.
    return resultado

## 4. Pruebas y ejecución real

Se conservan los casos y las comprobaciones de la práctica original. Los mensajes `OK` confirman que pasan y que la entrada permanece intacta. La tabla muestra las primeras 20 filas; `resultado` contiene la salida completa.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    variables = ["rooms_per_household", "bedrooms_per_room", "population_per_household"]
    caso = pd.DataFrame({
        "median_house_value": [1, 2, 3, 100],
        "rooms_per_household": [1, 2, 3, 4],
        "bedrooms_per_room": [4, 3, 2, 1],
        "population_per_household": [1, 1, 1, 2],
        "bedrooms_imputados": [False, False, False, True],
    }, index=[11, 3, 8, 20])
    original = caso.copy(deep=True)

    # Numpy sirve como comprobación independiente de las correlaciones de pandas.
    base = [np.corrcoef(caso[v], caso["median_house_value"])[0, 1] for v in variables]
    esperado = pd.DataFrame({
        "pearson_base": base,
        "pearson_sin_imputados": [1, -1, np.nan],
        "diferencia": [1 - base[0], -1 - base[1], np.nan],
    }, index=pd.Index(variables, name="variable"))
    comparar(funcion(caso), esperado)

    sin_imputados = caso.assign(bedrooms_imputados=False)
    esperado_sin = esperado.copy()
    esperado_sin["pearson_sin_imputados"] = base
    esperado_sin["diferencia"] = 0.0
    comparar(funcion(sin_imputados), esperado_sin)

    constante = caso.assign(median_house_value=1)
    comparar(funcion(constante), esperado * np.nan)
    comparar(caso, original)
    print("OK: signo de la diferencia, filtro de imputados y correlaciones indefinidas.")


probar(comparar_correlaciones)

# Comprobamos también que el dataset real permanezca intacto.
antes = datos.copy(deep=True)
resultado = comparar_correlaciones(datos)
comparar(datos, antes)
assert resultado.shape == (3, 3)
assert resultado[["pearson_base", "pearson_sin_imputados"]].abs().le(1).all().all()
np.testing.assert_allclose(
    resultado["diferencia"],
    resultado["pearson_sin_imputados"] - resultado["pearson_base"],
    equal_nan=True,
)
print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
display(resultado.head(20))

OK: signo de la diferencia, filtro de imputados y correlaciones indefinidas.
OK: dataset real. Resultado: 3 filas.


,pearson_base,pearson_sin_imputados,diferencia
variable,,,
rooms_per_household,0.151948,0.151344,-0.000604
bedrooms_per_room,-0.233303,-0.255880,-0.022577
population_per_household,-0.023737,-0.023639,0.000098


## Para repasar

Volvé a la práctica sin resolver e intentá escribir la función desde cero. Explicá qué representa cada fila de la salida, cómo se resuelven los casos límite y por qué el cálculo respeta la consigna.